In [1]:
# Install if needed
!pip install tensorflow

# Imports
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
print("Preparing IMDB sentiment dataset...")

VOCAB_LIMIT = 12000
SEQUENCE_LENGTH = 180

# Load dataset
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=VOCAB_LIMIT)

# Padding sequences
train_data = pad_sequences(train_data, maxlen=SEQUENCE_LENGTH, padding='pre')
test_data = pad_sequences(test_data, maxlen=SEQUENCE_LENGTH, padding='pre')

print("Train shape:", train_data.shape)

Preparing IMDB sentiment dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train shape: (25000, 180)


In [3]:
X_train_t = torch.LongTensor(train_data)
y_train_t = torch.FloatTensor(train_labels)

X_test_t = torch.LongTensor(test_data)
y_test_t = torch.FloatTensor(test_labels)

BATCH_SIZE = 128

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Running on:", device)

Running on: cpu


In [4]:
class TextRNN(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=128):
        super().__init__()

        self.embedding_layer = nn.Embedding(vocab_size, embed_size)

        self.rnn_layer = nn.RNN(
            input_size=embed_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.embedding_layer(x)

        rnn_out, hidden_state = self.rnn_layer(x)

        # Use last hidden state
        hidden_state = hidden_state[-1]

        out = self.output_layer(hidden_state)
        return out.squeeze()

In [5]:
model = TextRNN(VOCAB_LIMIT).to(device)

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 6

In [6]:
print("Training model...\n")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    correct_preds = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = torch.sigmoid(outputs)
        preds = (preds >= 0.5).float()
        correct_preds += (preds == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    accuracy = correct_preds / len(train_loader.dataset)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Accuracy: {accuracy*100:.2f}%")

Training model...

Epoch 1 | Loss: 0.6508 | Accuracy: 61.05%
Epoch 2 | Loss: 0.5591 | Accuracy: 71.18%
Epoch 3 | Loss: 0.5515 | Accuracy: 72.62%
Epoch 4 | Loss: 0.5542 | Accuracy: 71.94%
Epoch 5 | Loss: 0.4783 | Accuracy: 77.90%
Epoch 6 | Loss: 0.4550 | Accuracy: 79.03%


In [7]:
model.eval()

total_loss = 0
correct = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = loss_function(outputs, labels)

        total_loss += loss.item()

        preds = torch.sigmoid(outputs)
        preds = (preds >= 0.5).float()
        correct += (preds == labels).sum().item()

test_loss = total_loss / len(test_loader)
test_accuracy = correct / len(test_loader.dataset)

print("\nFinal Test Results:")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_accuracy*100:.2f}%")


Final Test Results:
Loss: 0.5011
Accuracy: 77.41%
